<a href="https://colab.research.google.com/github/DanLePoGo/untitled/blob/master/taquin_3x3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Importations
import random
import copy # Pour deepcopy()
import time

In [ ]:
#@title Classe Grille

NB_LIGNES = 3
NB_COLONNES = 3

class Grille:

    def __init__(self, valeurs=""):
        self._matrice = []
        self._position_X = (None, None)
        self._representation = ""
        compteur = 0
        for i in range(NB_LIGNES): #cree la matrice et trouve la position du X
          ligne = []
          for j in range(NB_COLONNES):
            if valeurs[compteur] == "X":
              self._position_X = (i,j)
            ligne.append(valeurs[compteur])
            compteur += 1
          self._matrice.append(ligne)

    def representation(self): #met a jour la representation sous la forme "12345678X"

        self._representation = ""
        for ligne in self._matrice:
          for valeur in ligne:
            self._representation += str(valeur)

    def __repr__(self):
        """
            Retourne la représentation de la grille sous la forme "12345678X".
            La case vide est représentée par un X
        """
        self.representation()
        return self._representation

    def coups_possibles(self):
        """
            Retourne une liste de tous les coups possibles dans la position actuelle
            de la grille
        """
        possibles = []
        if self._position_X[0] > 0:
          possibles.append("H")
        if self._position_X[0] < 2:
          possibles.append("B")
        if self._position_X[1] > 0:
          possibles.append("G")
        if self._position_X[1] < 2:
          possibles.append("D")
        return possibles

    def jouer_coup(self, coup):
        """
            Joue le coup dans la grille. Si le coup est impossible, on ne fait rien.
            Le coup est soit D, G, H ou B
        """

        if coup.upper() in self.coups_possibles(): #si le coup est possible, on l'effectue et on met a jour la position du X

          if coup.upper() == "H":
            self._matrice[self._position_X[0]][self._position_X[1]], self._matrice[self._position_X[0]-1][self._position_X[1]] = self._matrice[self._position_X[0]-1][self._position_X[1]], self._matrice[self._position_X[0]][self._position_X[1]]
            self._position_X = (self._position_X[0]-1, self._position_X[1])

          elif coup.upper() == "B":
            self._matrice[self._position_X[0]][self._position_X[1]], self._matrice[self._position_X[0]+1][self._position_X[1]] = self._matrice[self._position_X[0]+1][self._position_X[1]], self._matrice[self._position_X[0]][self._position_X[1]]
            self._position_X = (self._position_X[0]+1, self._position_X[1])

          elif coup.upper() == "G":
            self._matrice[self._position_X[0]][self._position_X[1]], self._matrice[self._position_X[0]][self._position_X[1]-1] = self._matrice[self._position_X[0]][self._position_X[1]-1], self._matrice[self._position_X[0]][self._position_X[1]]
            self._position_X = (self._position_X[0], self._position_X[1]-1)

          elif coup.upper() == "D":
            self._matrice[self._position_X[0]][self._position_X[1]], self._matrice[self._position_X[0]][self._position_X[1]+1] = self._matrice[self._position_X[0]][self._position_X[1]+1], self._matrice[self._position_X[0]][self._position_X[1]]
            self._position_X = (self._position_X[0], self._position_X[1]+1)


        else:
          print("Ce coup n'est pas possible")


    def afficher_grille(self): #affiche la grille

      for ligne in self._matrice:
        for valeur in ligne:
          if valeur == "X":
            print("  ", end="")
          else:
            print(valeur + " ", end="")
        print()

    def __eq__(self, other):
      return self._representation == other



In [ ]:
#@title Classe Resolveur

class Resolveur:

    def __init__(self, grille:Grille):
        self._grille = copy.deepcopy(grille)  # La grille à solutionner
        self._liste_deja = []
        self._solution = []

    def coup_inverse(self, coup): #retourne le coup inverse pour remettre la grille a son etat initial (optimisation)
        if coup == "H": return "B"
        if coup == "B": return "H"
        if coup == "G": return "D"
        if coup == "D": return "G"

    def solution_recursif(self, grille, profondeur_restante): #algoritme recursif qui trouve la solution

        if repr(grille) == "12345678X":
          return True
        elif profondeur_restante == 0:
          return False
        else:

          trouver = False
          for coup in grille.coups_possibles():

            grille.jouer_coup(coup) # Jouer un coup
            if repr(grille) in self._liste_deja: #si la repr a deja ete analysee, on remet la grille dans son etat initial
              grille.jouer_coup(self.coup_inverse(coup))

            else:
              self._liste_deja.append(repr(grille)) #stoque la repr dans une liste (optimisation)
              trouver = self.solution_recursif(grille, profondeur_restante - 1)
              grille.jouer_coup(self.coup_inverse(coup))

              if trouver:
                self._solution.append(coup)
                break

          return trouver

    def solutionner(self): #appelle la fonction solution recursif pour chaque profondeur jusqu'a quon trouve la solution
        """
            Procède à l'analyse de la grille. Retourne la solution sous la forme d'une liste de coups.
        """
        trouve = False
        profondeur = 1

        while not trouve:
          self._liste_deja = [repr(self._grille)]
          trouve = self.solution_recursif(self._grille, profondeur)
          profondeur += 1

        self._solution.reverse()
        return self._solution

In [ ]:
#@title Fonctions
# Vous devez placer ici vos fonctions s'il y a lieu

def est_valide(grille): #verifie si une grille est valide (nb d'inversions)

  valeurs_sans_X = grille.copy()
  valeurs_sans_X.remove("X") #copie de la liste melangee sans le X
  inversions = 0
  for i in range(len(valeurs_sans_X)-1): #compte le nombre d'inversion a faire dans la liste melangee
      for j in range(1 + i, len(valeurs_sans_X)):
        if valeurs_sans_X[j] < valeurs_sans_X[i]:
          inversions += 1

  if inversions % 2 == 0:
    return True

def generer_grille_aleatoire(): #genere et retourne une chaine de grille aleatoire avec une configuration valide
  valeurs = ["1", "2", "3", "4", "5", "6", "7", "8", "X"]
  valide = False

  while not valide:

    random.shuffle(valeurs)
    valide = est_valide(valeurs)

  return "".join(valeurs)

def description_grille(): #saisie la description d'une grille et verifie son format

  valide_inversion = False
  valide_contenu = False
  while not valide_inversion:
    while not valide_contenu: #verifie le format de la description avant de valider le nombre d'inversion, pour assurer que la fonction est_valide ne plante pas
      description = input("Entrez la description du problème: ")
      if len(description) == 9:
        valide_contenu = True
        for i in "12345678X":
          if i not in description:
            valide_contenu = False
      if not valide_contenu:
        print("Cette chaîne n'est pas valide")


    valide_inversion = est_valide(list(description))
    if not valide_inversion:
      valide_contenu = False #repartir la boucle, meme si le format etait valide
      print("Cette chaîne ne contient pas de solution")

  return description





In [ ]:
#@title Programme principal

# Cette fonction contient le programme
def __main__():

    grille_courante = Grille(generer_grille_aleatoire())
    quitter = False
    while not quitter: #afficher la grille et le menu tant que l'utilisateur n'a pas quitté

      grille_courante.afficher_grille()

      if repr(grille_courante) == "12345678X": #si on trouve la solution, on genere une nouvelle grille et on met a jour la representation
        print("Vous avez trouvé la solution!")
        print("Appuyez sur la touche retour pour générer une nouvelle grille")
        input()
        grille_courante = Grille(generer_grille_aleatoire())
        grille_courante.afficher_grille()

      print("Choisir parmi ces options:")
      print("Générer une nouvelle grille aléatoire (A)")
      print("Créer une grille à partir d'une description (C)")
      print("Déplacer la case vide (H, B, G, D)")
      print("Afficher la solution (S)")
      print("Tester une solution (T)")
      print("Quitter ce programme (Q)")
      choix = input("Entrez votre choix: ")
      print() #laisser un espace pour rendre l'execution plus claire

      if choix.lower() == "q":
        quitter = True

      elif choix.lower() == "a":
        grille_courante = Grille(generer_grille_aleatoire())

      elif choix.lower() == "c":
        grille_courante = Grille(description_grille())

      elif choix.lower() in ["h", "b", "g", "d"]:
        grille_courante.jouer_coup(choix)

      elif choix.lower() == "s":
        resous = Resolveur(Grille(repr(grille_courante)))
        avant = time.perf_counter()
        solution = "".join(resous.solutionner())
        apres = time.perf_counter()
        print(f"Voici la solution: {solution} trouvée en {apres-avant:.4f} secondes.")

      elif choix.lower() == "t":
        tester = input("Entrez la solution: ")
        for i in tester:
          if i.upper() in ["H", "B", "G", "D"]:
            grille_courante.jouer_coup(i)

      else:
        print("Option invalide. Choisir de nouveau")

    print("Merci d'avoir joué à ma version de taquin 3x3")
    print("Auteur: Dan Nguyen, DA: 2467699")


# Déclenchement du programme. Ne touchez pas à cette partie.
if __name__ == "__main__":
    __main__()

2 6 4 
8 1 5 
3 7   
Choisir parmi ces options:
Générer une nouvelle grille aléatoire (A)
Créer une grille à partir d'une description (C)
Déplacer la case vide (H, B, G, D)
Afficher la solution (S)
Tester une solution (T)
Quitter ce programme (Q)

5 2   
3 1 4 
7 8 6 
Choisir parmi ces options:
Générer une nouvelle grille aléatoire (A)
Créer une grille à partir d'une description (C)
Déplacer la case vide (H, B, G, D)
Afficher la solution (S)
Tester une solution (T)
Quitter ce programme (Q)

Voici la solution: GBGHDDBGGHDDBBGHHGBDBD trouvée en 15.6440 secondes.
5 2   
3 1 4 
7 8 6 
Choisir parmi ces options:
Générer une nouvelle grille aléatoire (A)
Créer une grille à partir d'une description (C)
Déplacer la case vide (H, B, G, D)
Afficher la solution (S)
Tester une solution (T)
Quitter ce programme (Q)

1 2 3 
4 5 6 
7   8 
Choisir parmi ces options:
Générer une nouvelle grille aléatoire (A)
Créer une grille à partir d'une description (C)
Déplacer la case vide (H, B, G, D)
Afficher la 